# Mufahras: build the search-by-meaning vectors

Embeds every verse (translation + Al-Mukhtasar explanation) and every English hadith with
**snowflake-arctic-embed-s**, the model chosen in rounds 1 and 2, and saves the vectors the app will ship.

The phone embeds each query with the model's **8-bit ONNX file**, so this notebook makes the text vectors two
ways and checks which matches the phone better:

- `q8`: the same 8-bit ONNX file, on the CPU (one text at a time, all cores in parallel); identical to what
  the phone and Transformers.js in CI compute
- `fp32`: the full-precision model on the GPU

Both are scored with queries embedded exactly as the phone will. Run all cells; download
`meaning-vectors.zip` at the end and unzip it into the repo's `data-src/vectors/`.

In [ ]:
%pip install -q -U sentence-transformers onnxruntime tokenizers huggingface_hub pandas
import os, torch; print(torch.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0), '|', os.cpu_count(), 'CPU cores')

## Settings

In [ ]:
MODEL_ID = 'Snowflake/snowflake-arctic-embed-s'
ONNX_FILE = 'onnx/model_quantized.onnx'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
MAX_TOKENS = 256
RECIPE = 1       # how texts are built (next cell); bump if that changes
LIMIT = None     # a number embeds only that many texts, for a quick test

import os
from huggingface_hub import model_info, hf_hub_download
REVISION = model_info(MODEL_ID).sha   # pinned: the app downloads this exact revision
print(MODEL_ID, REVISION)

## The texts (recipe 1)

- **Verse:** translation, a space, its Al-Mukhtasar explanation (verses in a group share the group's explanation).
- **Hadith:** the English as is.
- Both: footnote markers removed, whitespace collapsed.

Each vector is stored with a short hash of its text, so the app's build can tell which vectors are out of date
when the data changes, and re-embed only those.

In [ ]:
import json, re, hashlib, urllib.request, numpy as np, pandas as pd, time
BASE = 'https://omeiirr.github.io/mufahras/data/'
REPO = 'https://raw.githubusercontent.com/omeiirr/mufahras/main/'
get = lambda url: json.load(urllib.request.urlopen(url))
manifest = get(BASE + 'manifest.json')['chunks']
chunk = lambda name: get(BASE + manifest[name]['file'])
clean = lambda t: re.sub(r'\s+', ' ', re.sub('[\u00B9\u00B2\u00B3\u2070-\u209F]', '', t or '')).strip()

quran, tafsir = chunk('quran'), chunk('tafsir')
refs, texts = [], []
for s in quran['surahs']:
    for a in range(1, s['count'] + 1):
        i = s['start'] + a - 1
        t = tafsir['text'][i]
        if isinstance(t, int): t = tafsir['text'][t]
        refs.append(f"{s['n']}:{a}"); texts.append((clean(quran['en'][i]) + ' ' + clean(t)).strip())
for book in ['bukhari', 'muslim', 'abudawud', 'tirmidhi', 'nasai', 'ibnmajah']:
    for h in chunk(f'hadith-{book}')['hadiths']:
        if h[2] and h[2].strip():
            refs.append(f'{book} {h[0]}'); texts.append(clean(h[2]))
if LIMIT: refs, texts = refs[:LIMIT // 2] + refs[-LIMIT // 2:], texts[:LIMIT // 2] + texts[-LIMIT // 2:]
hashes = [hashlib.sha1(t.encode()).hexdigest()[:12] for t in texts]
queries = get(REPO + 'eval/meaning-queries.json')
print(len(texts), 'texts;', len(queries), 'queries')
print(texts[254][:300])

## `q8`: the 8-bit ONNX model, as the phone runs it

In [ ]:
import onnxruntime as ort
from tokenizers import Tokenizer
onnx_path = hf_hub_download(MODEL_ID, ONNX_FILE, revision=REVISION)
tok = Tokenizer.from_file(hf_hub_download(MODEL_ID, 'tokenizer.json', revision=REVISION))
tok.enable_truncation(MAX_TOKENS); tok.no_padding()
# One text per run, as Transformers.js does: the 8-bit model scales each run's values together, so batching
# would change a text's vector slightly (cosine ~0.995). One at a time, these match the phone exactly.
# Parallel instead: one thread per core, each running texts through the shared session.
opts = ort.SessionOptions(); opts.intra_op_num_threads = 1; opts.inter_op_num_threads = 1
sess = ort.InferenceSession(onnx_path, opts, providers=['CPUExecutionProvider'])
inputs = [i.name for i in sess.get_inputs()]
print(os.path.getsize(onnx_path) / 1e6, 'MB; inputs', inputs)

def embed_one(x):
    ids = np.array([tok.encode(x).ids], dtype=np.int64)
    feed = {'input_ids': ids, 'attention_mask': np.ones_like(ids)}
    if 'token_type_ids' in inputs: feed['token_type_ids'] = np.zeros_like(ids)
    cls = sess.run(None, feed)[0][0, 0]          # CLS pooling, as the model card and Transformers.js do
    return cls / np.linalg.norm(cls)

from concurrent.futures import ThreadPoolExecutor
def embed_q8(xs, progress=False):
    out = np.zeros((len(xs), 384), dtype=np.float32); t = time.time()
    with ThreadPoolExecutor(os.cpu_count()) as pool:
        for i, v in enumerate(pool.map(embed_one, xs)):
            out[i] = v
            if progress and i % 5000 == 0: print(f'  {i}/{len(xs)} {time.time() - t:.0f}s', flush=True)
    return out

t = time.time()
docs_q8 = embed_q8(texts, progress=True)
print(f'q8: {len(texts)} texts in {time.time() - t:.0f}s')

## `fp32`: the full-precision model on the GPU

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(MODEL_ID, revision=REVISION, device='cuda' if torch.cuda.is_available() else 'cpu')
st.max_seq_length = MAX_TOKENS
t = time.time()
docs_fp32 = st.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
print(f'fp32: {time.time() - t:.0f}s')
agree = (docs_q8 * docs_fp32).sum(1)
print(f'q8 vs fp32, same text: mean cosine {agree.mean():.4f}, lowest {agree.min():.4f}')  # near 1 = pooling matches

## Which text vectors match the phone's queries best?

In [ ]:
def int8(v, scale): return np.clip(np.round(v * scale), -127, 127).astype(np.int8)
def score(docs, qv):
    """hit@10 counts an original answer or an accepted alternative; strict only the original answers."""
    top = np.argsort(-(qv @ docs.T), axis=1)[:, :10]
    hit = strict = mrr = 0
    for j, q in enumerate(queries):
        got = [refs[i] for i in top[j]]; ok = set(q['expect']) | set(q.get('also', {}))
        first = next((k for k, r in enumerate(got) if r in ok), None)
        hit += first is not None; strict += any(r in q['expect'] for r in got); mrr += 0 if first is None else 1 / (first + 1)
    n = len(queries)
    return {'hit@10': round(100 * hit / n), 'strict hit@10': round(100 * strict / n), 'MRR': round(mrr / n, 3)}

q_phone = embed_q8([QUERY_PREFIX + q['query'] for q in queries])
q_fp32 = st.encode([QUERY_PREFIX + q['query'] for q in queries], normalize_embeddings=True, convert_to_numpy=True)
scales = {name: 127 / np.abs(d).max() for name, d in (('q8', docs_q8), ('fp32', docs_fp32))}
rows = []
for name, d in (('q8', docs_q8), ('fp32', docs_fp32)):
    d8 = int8(d, scales[name]).astype(np.float32)
    rows.append({'text vectors': name, 'queries': 'phone (q8)', 'stored as': 'float32', **score(d, q_phone)})
    rows.append({'text vectors': name, 'queries': 'phone (q8)', 'stored as': 'int8', **score(d8, q_phone)})
    rows.append({'text vectors': name, 'queries': 'fp32', 'stored as': 'float32', **score(d, q_fp32)})
pd.DataFrame(rows)

## Save

In [ ]:
os.makedirs('vectors-out', exist_ok=True)
for name, d in (('q8', docs_q8), ('fp32', docs_fp32)):
    int8(d, scales[name]).tofile(f'vectors-out/arctic-s.{name}.i8')
json.dump({
    'model': MODEL_ID, 'revision': REVISION, 'onnx': ONNX_FILE, 'queryPrefix': QUERY_PREFIX, 'maxTokens': MAX_TOKENS,
    'pooling': 'cls', 'dims': 384, 'recipe': RECIPE, 'count': len(refs),
    'scale': {k: float(v) for k, v in scales.items()},   # int8 value = round(float × scale)
    'refs': refs, 'hashes': hashes,
}, open('vectors-out/index.json', 'w'))
!cd vectors-out && zip -q ../meaning-vectors.zip * && cd .. && ls -lh vectors-out meaning-vectors.zip